# Assignment 3: LLMs and Machine Learning

---

## Submission Instructions

Submit only a link to the folder for Assignment 3 in your personal GitHub repository. Within the repository, you should have a Jupyter notebook file titled e.g. `assignment3.ipynb` or something similar, placed inside the `assignments/assignment3/` folder.

Make sure the repository is public.

**Submissions must be made using a GitHub repository. Submissions that do not follow this instruction will receive 0 points.**

**Late submissions are not accepted as the peer review system does not allow adding submissions past the deadline. Submit your work early to not miss the deadline!**

## Code Quality

Write your code so that it is pleasant to read and easy to understand. This includes:

- Use descriptive variable and function names.
- Add brief comments where the logic is not immediately obvious.
- Keep your notebook organized with clear separation between tasks.
- Print out your answers so that the peer reviewer can see the results. Use the `df.head()` when asked to print the top  5 lines. To print a better looking DataFrame, consider also using `display()` instead of `print()`.
- Divide the code into logical chunks. At minimum, use separate cells per task, and when reasonable, separate cells for subtasks.
- Remember to in the end rerun all code from the beginning to end of the notebook to ensure each cell runs without error

## Visualizations

In the visualizations always include enough information that the plot can be understood independently. This includes:

- Labels for both axes
- A descriptive title

## Statement of use of AI

Include a brief statement describing how and which AI was used (or if no AI was used) in completing the assignment. This could be a markdown cell with a couple of sentences. As a reminder, AI use is permitted in the assignments, but it is advisable to try to complete the tasks as far as possible without and to make sure you understand the code that AI produced when using it.

AI was used to help me refine my prompts. AI also gave me examples on how to use the various sci-kit learn functions. 
```python
# an example
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier

# Load data
iris = load_iris()
X_train, X_test, y_train, y_test = train_test_split(iris.data, iris.target, test_size=0.2)

# KNN requires scaling for accurate distance math
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Initialize and fit
knn = KNeighborsClassifier(n_neighbors=5)
knn.fit(X_train_scaled, y_train)

print(f"KNN Accuracy: {knn.score(X_test_scaled, y_test):.2f}")
```

## Grading

This assignment is worth 10 points. Task 0 is worth 1 point, and tasks 1-2 are worth 2 points and task 3 is worth 5 points.

Points are given only for code that runs. If the code does not run, the task (or subtask if code for a task is divided into multiple cells) will automatically receive 0 points even if the code is almost correct.

### Penalties

- **-2 points per task** where AI-generated (hallucinated) data is used instead of the actual data provided in the task or retrieved from the specified source. The assignment requires working with real data, not made-up values!
- **-3 points** if an API key is included in the submission notebook or anywhere in the GitHub repository. Store your keys in a `.env` file and add `.env` to your `.gitignore`.
- **-1 point** if the Jupyter Notebook is overall messy and not structured well (e.g. if all tasks are completed within one cell, if answers are difficult to find due to too much irrelevant printed output).
- **-1 point** if there is no statement of AI use. If you did not use AI, report that you did not use AI.

### Editing the submission after the deadline

- Editing the assignment submission during the evaluation phase is forbidden. Thus, after the solution has been released, do not make any further changes to the notebook until you have received a grade. If you accidentally leaked an API key, revoke the key immediately. Other **changes to the submission are considered cheating, and will result in 0 points for both the assignment and peer review**.

---

## Tasks

### Task 0: Setting up Ollama (1p)

a) Set up Ollama and connect to it using either openAI's API or Ollama's own API. 

b) Load the 270m parameter version of the [gemma3](https://ollama.com/library/gemma3) model and test it with any prompt.

c) Load the 4b parameter version of the [gemma3](https://ollama.com/library/gemma3) and test it with any prompt. If running the 4b version is too slow, you can use the 1b version instead.

In [6]:
pip install ollama

Note: you may need to restart the kernel to use updated packages.


In [ ]:
# a)
import ollama
from dotenv import load_dotenv
import os
from ollama import Client


load_dotenv()
OLLAMA_API_KEY = os.getenv("API_KEY")

client = Client(
    host="https://ollama.com",
    headers={'Authorization': 'Bearer ' + OLLAMA_API_KEY}
)

messages = [
    {
        'role':'user',
        'content': 'What is 1+1?',
    },
]

# b)
print('c)')
for part in client.chat(model='gemma3:4b-cloud', messages=messages, stream=True):
    print(part['message']['content'], end='', flush=True)


c)
1 + 1 = 2


In [8]:
# b)
print('b)')
!ollama pull gemma3:270m

b)


In [9]:
# b
print('b)')
local_client = Client(host='http://localhost:11434')

for part in local_client.chat(model='gemma3:270m', messages=messages, stream=True):
    print(part['message']['content'], end='', flush=True)

b)
1 + 1 = 2


### Task 1: Text classification with Ollama (2p)

The `data/emails.csv` file contains 12 email headlines, with 4 spam emails, 4 legitimate work emails and 4 vague emails that are hard to classify based on the title alone. Use this dataset for all subtasks in this task.

a) Make a function for classifying emails (based on the headlines) as spam, work or unknown. The function should return only the classification and nothing else. (0.5p)

b) Use the smaller gemma3 (270m) to classify the emails using the function created in part a. (0.5p)

c) Use larger gemma3 (4b) to classify the emails using the function created in part a). In separate markdown cell, write a brief comment comparing the results of parts b) and c). (0.5p)

d) Write a script that repeats b) and c) 3 times, storing the results for both models separately. For both models, put the results as columns into a new DataFrame that also contains the headlines so that it is easy to compare how the output varied across runs for both models. Comment if there were differences and explain why this happened. (0.5p)

In [10]:
import csv
import os
print('a')

directory = '.\data'
filename = 'emails.csv'
filepath = os.path.join(directory, filename)
header_list = []

with open(filepath, mode='r') as f:
    csv_file = csv.reader(f)
    # to skip the header of this csv file
    next(csv_file)
    for header in csv_file:
        header_list.append(header)
    # print(header_list)
     

def classifier(client_used, model_used, emails):
    classifier_client = client_used
    result = []
    for email in emails:
        # use for checking
        # print(email)
        classifier_messages = [{'role':'system', 'content':'You are an email classifier. Categorize the following email into exactly one of these three labels: spam, legitimate, or vague. Output the label in lowercase only. Do not include introductory text, explanations, or punctuation.'}]
        temp = {'role':'user'}
        temp['content'] = f'email: {email}'
        classifier_messages.append(temp)
        msg=''
        for part in classifier_client.chat(model=model_used, messages=classifier_messages, stream=True):
            msg += part['message']['content']
        # check the mes
        # print(msg)
        result.append(msg)
    return result

a


In [11]:
print('b')
result_270 = classifier(local_client, 'gemma3:270m', header_list)
result_270 = [ans.rstrip() for ans in result_270]
print(result_270)

b
['spam', 'spam', 'spam', 'spam', 'spam', 'spam', 'spam', 'spam', 'spam', 'spam', 'spam', 'spam']


In [12]:
print('c')
result_4b = classifier(client, 'gemma3:4b-cloud', header_list)
result_4b = [ans.rstrip() for ans in result_4b]
print(result_4b)

c
['spam', 'spam', 'spam', 'spam', 'legitimate', 'legitimate', 'legitimate', 'legitimate', 'vague', 'vague', 'spam', 'vague']


In [13]:
print('d')
store_270 = []
store_4b = []

for _ in range(3):
    result_270 = classifier(local_client, 'gemma3:270m', header_list)
    result_270 = [ans.rstrip() for ans in result_270]
    store_270.append(result_270)
    result_4b = classifier(client, 'gemma3:4b-cloud', header_list)
    result_4b = [ans.rstrip() for ans in result_4b]
    store_4b.append(result_4b)

d


In [14]:
print(header_list)

[['URGENT: Your account will be suspended within 24 hours'], ['Congratulations! You have won a 1000€ gift card', ' claim now'], ['Hot singles in your area are waiting to meet you tonight'], ['Re: Inheritance transfer of 4.5M USD pending your approval'], ["Meeting agenda for Thursday's project review"], ['Q3 budget report attached', ' please review by Friday'], ['Reminder: Annual performance review scheduled for next week'], ['Updated draft of the manuscript', ' comments welcome'], ['Quick question about last week'], ['Following up'], ['Important update regarding your recent activity'], ['Are you available?']]


In [15]:
import pandas as pd

column_names = [i[0] for i in header_list]
df270 = pd.DataFrame(store_270, columns=column_names)
df4b = pd.DataFrame(store_4b, columns=column_names)

In [16]:
display(df270)

,URGENT: Your account will be suspended within 24 hours,Congratulations! You have won a 1000€ gift card,Hot singles in your area are waiting to meet you tonight,Re: Inheritance transfer of 4.5M USD pending your approval,Meeting agenda for Thursday's project review,Q3 budget report attached,Reminder: Annual performance review scheduled for next week,Updated draft of the manuscript,Quick question about last week,Following up,Important update regarding your recent activity,Are you available?
0,spam,spam,spam,spam,spam,spam,spam,spam,spam,spam,spam,spam
1,spam,spam,spam,spam,spam,spam,spam,spam,spam,spam,spam,spam
2,spam,spam,spam,spam,spam,spam,spam,spam,spam,spam,spam,spam


In [17]:
display(df4b)

,URGENT: Your account will be suspended within 24 hours,Congratulations! You have won a 1000€ gift card,Hot singles in your area are waiting to meet you tonight,Re: Inheritance transfer of 4.5M USD pending your approval,Meeting agenda for Thursday's project review,Q3 budget report attached,Reminder: Annual performance review scheduled for next week,Updated draft of the manuscript,Quick question about last week,Following up,Important update regarding your recent activity,Are you available?
0,spam,spam,spam,spam,legitimate,legitimate,legitimate,legitimate,vague,vague,spam,vague
1,spam,spam,spam,spam,legitimate,legitimate,legitimate,legitimate,vague,vague,spam,vague
2,spam,spam,spam,spam,legitimate,legitimate,legitimate,legitimate,vague,vague,spam,vague


Part d) The output remained the same for both models where the one which runs on the 270m parameter outputted the same answer for everything. This could be due to the model not understanding the question correctly every time. The model running on 4b parameters outputted the same answer for each of one of the three run. It could be that the email header is very clearly separated into the three categories so the model will generate the same answer every time. 

### Task 2: Sentiment analysis with Ollama (2p)

The `data/news.csv` file contains 10 fictional financial news headlines. Use it for all subtasks in this task.

a) Make a function for classifying the texts in the provided dataset based on the topic (earnings, mergers, regulation, macroeconomics) and for determining the sentiment of the news (positive, negative, neutral). The function should return the class and sentiment in JSON format. (1p)

b) Use gemma3 (4b) to classify and provide the sentiment for each row of the provided dataset, inserting them into a new DataFrame that contains both the original headlines as well as topic and sentiment. (0.5p)

c) Give the same data and prompt to a browser based LLM (e.g. ChatGPT, LeChat, Claude or Gemini) and ask it to provide the topic and sentiment, giving it the same options. Paste the results into a markdown cell. Compare the results of b) and c), which one is more accurate and why? (0.5p)

In [18]:
filename2 = 'news.csv'
directory2 = './data'
filepath2 = os.path.join(directory2, filename2)


news_list = []
with open(filepath2, 'r') as f:
    news = csv.reader(f)
    next(news)
    for n in news:
        news_list += n

In [19]:
# realized there was an error so I changed the news.csv the commas in the original csv file broke the csv reader causing there to be additional header
print(news_list)

['Nordion Industries beats Q1 earnings estimates as cloud revenue surges 28%', 'Helvora Pharmaceuticals misses earnings forecast amid weak generics demand', 'Aurelis Bank reports steady quarterly profit, in line with analyst expectations', 'Veridyne Logistics to acquire rival Trantec in 4.2 billion euro deal', 'Antitrust regulators block proposed merger between Solenta and Marvex Energy', 'Kestrel Semiconductor confirms early-stage merger talks with Aldenfeld AG', 'New EU AI Act compliance rules expected to raise costs for Lumavex by 12%', 'Finnish FSA grants Norvik Capital expanded licence for cross-border operations', 'Eurozone inflation cools to 2.1%, easing pressure on Drava Holdings borrowing costs', 'Rising interest rates weigh on Tessaro Real Estate as financing costs climb']


In [20]:
print('a')
import re
import json
def news_classifier(client_used, model_used, news):
    result = []
    for n in news:
        # use for checking
        # print(email)
        classifier_messages = [{'role':'system', 'content':'''You are an expert financial analyst. Analyze the following news text and determine its primary topic and sentiment.
                            Instructions:
                            Classify the Topic: Choose exactly one from: [earnings, mergers, regulation, macroeconomics].
                            Determine Sentiment: Choose exactly one from: [positive, negative, neutral].
                            Output Format: Return the result strictly in raw JSON object only with the keys 'class' and 'sentiment'. Do not include Markdown code blocks, backticks (```), or any preamble/explanation. 
                                '''}]
        temp = {'role':'user'}
        temp['content'] = f'text: {n}'
        classifier_messages.append(temp)
        msg=''
        for part in client_used.chat(model=model_used, messages=classifier_messages, stream=True):
            msg += part['message']['content']
        # check the mes
        # print(msg)
        # have to clean because the prompting did not work
        clean_str = re.sub(r'```json|```', '', msg).strip()
        cleaned_msg = json.loads(clean_str)
        result.append(cleaned_msg)
    return result

a


In [21]:
print('b')
news_4b = news_classifier(client, 'gemma3:4b-cloud', news_list)

b


In [22]:
df_news_header = pd.DataFrame({'temp': news_list})

In [23]:
df_news = pd.DataFrame(news_4b)
df_news['news header'] = df_news_header['temp']
display(df_news)


,class,sentiment,news header
0,earnings,positive,Nordion Industries beats Q1 earnings estimates...
1,earnings,negative,Helvora Pharmaceuticals misses earnings foreca...
2,earnings,positive,"Aurelis Bank reports steady quarterly profit, ..."
3,mergers,positive,Veridyne Logistics to acquire rival Trantec in...
4,regulation,negative,Antitrust regulators block proposed merger bet...
5,mergers,neutral,Kestrel Semiconductor confirms early-stage mer...
6,regulation,negative,New EU AI Act compliance rules expected to rai...
7,regulation,positive,Finnish FSA grants Norvik Capital expanded lic...
8,macroeconomics,neutral,"Eurozone inflation cools to 2.1%, easing press..."
9,macroeconomics,negative,Rising interest rates weigh on Tessaro Real Es...


From Chatgpt and this seems more accurate based on my understanding of the news header. This is likely because the online model is newer and has more parameters thus is more likely to be able to detect the class and sentiment.<br>
{"class":"earnings","sentiment":"positive"},<br>
{"class":"earnings","sentiment":"negative"},<br>
{"class":"earnings","sentiment":"neutral"},<br>
{"class":"mergers","sentiment":"positive"},<br>
{"class":"mergers","sentiment":"negative"},<br>
{"class":"mergers","sentiment":"neutral"},<br>
{"class":"regulation","sentiment":"negative"},<br>
{"class":"regulation","sentiment":"positive"},<br>
{"class":"macroeconomics","sentiment":"positive"},<br>
{"class":"macroeconomics","sentiment":"negative"}

### Task 3: Supervised machine learning (5p)

For this task, use a subset of the [Bank Marketing](https://archive.ics.uci.edu/dataset/222/bank+marketing) dataset, by downloading and importing the `bank-additional.csv` from the UCI repository. You can find a description of the dataset behind the link.

The goal is to predict whether a prospective customer will subscribe to a term deposit (variable y).

a) Import the dataset and conduct exploratory data analysis on it. (1p)

b) Preprocess the data using the appropriate methods as described in the course materials. (1p)

c) Determine whether this is a classification or regression task. Choose three different machine learning algorithms from scikit-learn and explain briefly why you chose them. For each of the selected algorithsm, train and a model and iteratively adjust the hyperparameters until you no longer manage to improve the performance. (1p)

d) Compare using train, validation and test set split versus using cross-validation. Which one performs better? (1p)

e) Report and evaluate the performance of the models using several of the metrics provided in the course, and explain which model is the best for the task and why. (1p)


   Input variables:
   ### bank client data:
   1 - age (numeric)<br>
   2 - job : type of job (categorical: "admin.","blue-collar","entrepreneur","housemaid","management","retired","self-employed","services","student","technician","unemployed","unknown")<br>
   3 - marital : marital status (categorical: "divorced","married","single","unknown"; note: "divorced" means divorced or widowed)<br>
   4 - education (categorical: "basic.4y","basic.6y","basic.9y","high.school","illiterate","professional.course","university.degree","unknown")<br>
   5 - default: has credit in default? (categorical: "no","yes","unknown")<br>
   6 - housing: has housing loan? (categorical: "no","yes","unknown")<br>
   7 - loan: has personal loan? (categorical: "no","yes","unknown")<br>
   ### related with the last contact of the current campaign:
   8 - contact: contact communication type (categorical: "cellular","telephone") <br>
   9 - month: last contact month of year (categorical: "jan", "feb", "mar", ..., "nov", "dec")<br>
  10 - day_of_week: last contact day of the week (categorical: "mon","tue","wed","thu","fri")<br>
  11 - duration: last contact duration, in seconds (numeric). Important note:  this attribute highly affects the output target (e.g., if duration=0 then y="no"). Yet, the duration is not known before a call is performed. Also, after the end of the call y is obviously known. Thus, this input should only be included for benchmark purposes and should be discarded if the intention is to have a realistic predictive model.<br>
  ### other attributes:
  12 - campaign: number of contacts performed during this campaign and for this client (numeric, includes last contact)<br>
  13 - pdays: number of days that passed by after the client was last contacted from a previous campaign (numeric; 999 means client was not previously contacted)<br>
  14 - previous: number of contacts performed before this campaign and for this client (numeric)<br>
  15 - poutcome: outcome of the previous marketing campaign (categorical: "failure","nonexistent","success")<br>
  ### social and economic context attributes
  16 - emp.var.rate: employment variation rate - quarterly indicator (numeric)<br>
  17 - cons.price.idx: consumer price index - monthly indicator (numeric)  <br>   
  18 - cons.conf.idx: consumer confidence index - monthly indicator (numeric)  <br>   
  19 - euribor3m: euribor 3 month rate - daily indicator (numeric)<br>
  20 - nr.employed: number of employees - quarterly indicator (numeric)<br>

  ### Output variable (desired target):
  21 - y - has the client subscribed a term deposit? (binary: "yes","no") <br>

Missing Attribute Values: There are several missing values in some categorical attributes, all coded with the "unknown" label. These missing values can be treated as a possible class label or using deletion or imputation techniques. 


In [24]:
filename3 = 'bank-additional.csv'
directory3 = './data'
filepath3 = os.path.join(directory3, filename3)
# with open(filepath3, 'r') as f:
#     dataset = csv.reader(f)
df3 = pd.read_csv(filepath3, delimiter=';')
display(df3)

,age,job,marital,education,default,housing,loan,contact,month,day_of_week,...,campaign,pdays,previous,poutcome,emp.var.rate,cons.price.idx,cons.conf.idx,euribor3m,nr.employed,y
0,30,blue-collar,married,basic.9y,no,yes,no,cellular,may,fri,...,2,999,0,nonexistent,-1.8,92.893,-46.2,1.313,5099.1,no
1,39,services,single,high.school,no,no,no,telephone,may,fri,...,4,999,0,nonexistent,1.1,93.994,-36.4,4.855,5191.0,no
2,25,services,married,high.school,no,yes,no,telephone,jun,wed,...,1,999,0,nonexistent,1.4,94.465,-41.8,4.962,5228.1,no
3,38,services,married,basic.9y,no,unknown,unknown,telephone,jun,fri,...,3,999,0,nonexistent,1.4,94.465,-41.8,4.959,5228.1,no
4,47,admin.,married,university.degree,no,yes,no,cellular,nov,mon,...,1,999,0,nonexistent,-0.1,93.200,-42.0,4.191,5195.8,no
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4114,30,admin.,married,basic.6y,no,yes,yes,cellular,jul,thu,...,1,999,0,nonexistent,1.4,93.918,-42.7,4.958,5228.1,no
4115,39,admin.,married,high.school,no,yes,no,telephone,jul,fri,...,1,999,0,nonexistent,1.4,93.918,-42.7,4.959,5228.1,no
4116,27,student,single,high.school,no,no,no,cellular,may,mon,...,2,999,1,failure,-1.8,92.893,-46.2,1.354,5099.1,no
4117,58,admin.,married,high.school,no,no,no,cellular,aug,fri,...,1,999,0,nonexistent,1.4,93.444,-36.1,4.966,5228.1,no


In [25]:
print('a')
df3.describe()
# although duration max and mean has a rather large difference, the durations is in seconds so it seems still reasonable

a


,age,duration,campaign,pdays,previous,emp.var.rate,cons.price.idx,cons.conf.idx,euribor3m,nr.employed
count,4119.000000,4119.000000,4119.000000,4119.000000,4119.000000,4119.000000,4119.000000,4119.000000,4119.000000,4119.000000
mean,40.113620,256.788055,2.537266,960.422190,0.190337,0.084972,93.579704,-40.499102,3.621356,5166.481695
std,10.313362,254.703736,2.568159,191.922786,0.541788,1.563114,0.579349,4.594578,1.733591,73.667904
min,18.000000,0.000000,1.000000,0.000000,0.000000,-3.400000,92.201000,-50.800000,0.635000,4963.600000
25%,32.000000,103.000000,1.000000,999.000000,0.000000,-1.800000,93.075000,-42.700000,1.334000,5099.100000
50%,38.000000,181.000000,2.000000,999.000000,0.000000,1.100000,93.749000,-41.800000,4.857000,5191.000000
75%,47.000000,317.000000,3.000000,999.000000,0.000000,1.400000,93.994000,-36.400000,4.961000,5228.100000
max,88.000000,3643.000000,35.000000,999.000000,6.000000,1.400000,94.767000,-26.900000,5.045000,5228.100000


In [26]:
# based on the description in the file the categorical columns with unknown datas are
categorical_cols = ['job', 'marital', 'education', 'default', 'housing', 'loan']
for col in categorical_cols:
    print(df3[col].value_counts().head(10))

job
admin.           1012
blue-collar       884
technician        691
services          393
management        324
retired           166
self-employed     159
entrepreneur      148
unemployed        111
housemaid         110
Name: count, dtype: int64
marital
married     2509
single      1153
divorced     446
unknown       11
Name: count, dtype: int64
education
university.degree      1264
high.school             921
basic.9y                574
professional.course     535
basic.4y                429
basic.6y                228
unknown                 167
illiterate                1
Name: count, dtype: int64
default
no         3315
unknown     803
yes           1
Name: count, dtype: int64
housing
yes        2175
no         1839
unknown     105
Name: count, dtype: int64
loan
no         3349
yes         665
unknown     105
Name: count, dtype: int64


In [27]:
# we shall just replace them with the mode althought there could be better ways to replace them
print('b')
for col in categorical_cols:
    # mode returns a series so use [0] to get the first value
    col_mode = df3[col].mode()[0]
    df3[col] = df3[col].replace('unknown', col_mode)

b


In [28]:
# double check whether there are still unknown left
categorical_cols = ['job', 'marital', 'education', 'default', 'housing', 'loan']
for col in categorical_cols:
    print(df3[col].value_counts().head())

job
admin.         1051
blue-collar     884
technician      691
services        393
management      324
Name: count, dtype: int64
marital
married     2520
single      1153
divorced     446
Name: count, dtype: int64
education
university.degree      1431
high.school             921
basic.9y                574
professional.course     535
basic.4y                429
Name: count, dtype: int64
default
no     4118
yes       1
Name: count, dtype: int64
housing
yes    2280
no     1839
Name: count, dtype: int64
loan
no     3454
yes     665
Name: count, dtype: int64


c) Determine whether this is a classification or regression task. Choose three different machine learning algorithms from scikit-learn and explain briefly why you chose them. For each of the selected algorithsm, train and a model and iteratively adjust the hyperparameters until you no longer manage to improve the performance. (1p)

In [29]:
print('c) This is a classification task as the value that we are predicting is categorical. ')
print('The first algorithm that I choose is k-nearest neighbour because it can be visualized if needed')
print('The second algorithm that I choose is random forest because it works well with non-linear data')
print('The second algorithm that I choose is logisic regression because it is fast to train')

c) This is a classification task as the value that we are predicting is categorical. 
The first algorithm that I choose is k-nearest neighbour because it can be visualized if needed
The second algorithm that I choose is random forest because it works well with non-linear data
The second algorithm that I choose is logisic regression because it is fast to train


We will focus on these data as they seem to affect the outcome the most:<br>
   1 - age (numeric)<br>
   2 - job : type of job (categorical: "admin.","blue-collar","entrepreneur","housemaid","management","retired","self-employed","services","student","technician","unemployed","unknown")<br>
   3 - marital : marital status (categorical: "divorced","married","single","unknown"; note: "divorced" means divorced or widowed)<br>
   4 - education (categorical: "basic.4y","basic.6y","basic.9y","high.school","illiterate","professional.course","university.degree","unknown")<br>
   5 - default: has credit in default? (categorical: "no","yes","unknown")<br>
   6 - housing: has housing loan? (categorical: "no","yes","unknown")<br>
   7 - loan: has personal loan? (categorical: "no","yes","unknown")<br>

In [30]:
print('c) train model and adjust hyperparameter.')
print('KNN')


from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, MinMaxScaler, OrdinalEncoder
from sklearn.model_selection import GridSearchCV

# orders from lowest education level to highest education level
ordinal_order = ["illiterate", "basic.4y", "basic.6y", "basic.9y", 
                 "high.school", "professional.course", "university.degree"]

target = df3['y']

# have to convert the categorical data to numerical, column transformer helps to create transformer objects to be applied to the columns specified.
preprocessor = ColumnTransformer(
    transformers=[
        # to ensure the age does not have too high weightage
        ('num', MinMaxScaler(), ['age']),
        ('ord', OrdinalEncoder(categories=[ordinal_order]), ['education']),
        ('nom', OneHotEncoder(sparse_output=False), ['job', 'marital']),
        ('bin', OneHotEncoder(drop='if_binary'), ['default', 'housing', 'loan'])
    ])
features = df3[['age', 'job', 'marital', 'education', 'default', 'housing', 'loan']]
features_processed = preprocessor.fit_transform(features)

X_train_k, X_test_k, y_train_k, y_test_k = train_test_split(features_processed, target, test_size=0.2)

# KNN requires scaling for accurate distance math
scaler = StandardScaler()
X_train_scaled_k = scaler.fit_transform(X_train_k)
X_test_scaled_k = scaler.transform(X_test_k)

# use gridsearch cv
test_value = {'n_neighbors': range(1, 31)}

# Use GridSearchCV to find the best k
grid_search = GridSearchCV(KNeighborsClassifier(), test_value)
grid_search.fit(X_train_scaled_k, y_train_k)

print(f"Best k: {grid_search.best_params_['n_neighbors']}") 
print('After doing part E, there is a problem where the model predicted all "no" so we have to manually check whether the model is outputting yes while checking the accuracy')
print('N = 6')
knn = KNeighborsClassifier(n_neighbors=6)
knn.fit(X_train_scaled_k, y_train_k)
print('The accuracy for KNN:', knn.score(X_test_scaled_k, y_test_k))
y_pred = knn.predict(X_test_scaled_k)
display(pd.Series(y_pred).value_counts())
print()
print('N = 7')
knn = KNeighborsClassifier(n_neighbors=7)
knn.fit(X_train_scaled_k, y_train_k)
print('The accuracy for KNN:', knn.score(X_test_scaled_k, y_test_k))
y_pred = knn.predict(X_test_scaled_k)
display(pd.Series(y_pred).value_counts())
print()
print('N = 8')
knn = KNeighborsClassifier(n_neighbors=8)
knn.fit(X_train_scaled_k, y_train_k)
print('The accuracy for KNN:', knn.score(X_test_scaled_k, y_test_k))
y_pred = knn.predict(X_test_scaled_k)
display(pd.Series(y_pred).value_counts())
print()
print('N = 9')
knn = KNeighborsClassifier(n_neighbors=9)
knn.fit(X_train_scaled_k, y_train_k)
print('The accuracy for KNN:', knn.score(X_test_scaled_k, y_test_k))
y_pred = knn.predict(X_test_scaled_k)
display(pd.Series(y_pred).value_counts())
print()

c) train model and adjust hyperparameter.
KNN


Best k: 12
After doing part E, there is a problem where the model predicted all "no" so we have to manually check whether the model is outputting yes while checking the accuracy
N = 6
The accuracy for KNN: 0.8798543689320388


no     820
yes      4
Name: count, dtype: int64


N = 7
The accuracy for KNN: 0.8786407766990292


no     815
yes      9
Name: count, dtype: int64


N = 8
The accuracy for KNN: 0.8822815533980582


no     822
yes      2
Name: count, dtype: int64


N = 9
The accuracy for KNN: 0.8822815533980582


no     822
yes      2
Name: count, dtype: int64

In [42]:
# finding the accuracy I decided to go with k = 7, as the accuracy is roughly the same
knn = KNeighborsClassifier(n_neighbors=7)
knn.fit(X_train_scaled_k, y_train_k)
print('The accuracy for KNN:', knn.score(X_test_scaled_k, y_test_k))

The accuracy for KNN: 0.8786407766990292


In [32]:
print('Random forest classifier')
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import RandomizedSearchCV

# transforming the data into something the model can use
tree_preprocessor = ColumnTransformer(
    transformers=[
        # No normalization needed for random tree
        ('num', 'passthrough', ['age']), 
        ('ord', OrdinalEncoder(categories=[ordinal_order]), ['education']),
        # change to ordinal since it will make the tree more efficient instead of having the tree split into more branches
        ('nom', OrdinalEncoder(), ['job', 'marital']), 
        ('bin', OrdinalEncoder(), ['default', 'housing', 'loan'])
    ])
features_processed_tree = tree_preprocessor.fit_transform(features)
X_train_rf, X_test_rf, y_train_rf, y_test_rf = train_test_split(features_processed_tree, target, test_size=0.2)

# We shall use randomizedsearchcv since it will be alot faster, the list of value is changed based on the previous results of my testing
params = {
    'n_estimators': [50, 100, 300, 400],
    'max_depth': [None, 20, 30, 40],
    'min_samples_split': [2, 5, 8, 10],
    'bootstrap': [True]
}

random_search = RandomizedSearchCV(RandomForestClassifier(class_weight='balanced'), param_distributions=params, random_state=1)
random_search.fit(X_train_rf, y_train_rf)
print(f'Best parameters: {random_search.best_params_}')

Random forest classifier
Best parameters: {'n_estimators': 300, 'min_samples_split': 2, 'max_depth': 40, 'bootstrap': True}


In [44]:
# optimal random random forest hyperparameter after testing
rf = RandomForestClassifier(n_estimators=300, max_depth=40, min_samples_split=2, random_state=1, bootstrap=True, class_weight='balanced')
rf.fit(X_train_rf, y_train_rf)
print(f"Random Forest Accuracy: {rf.score(X_test_rf, y_test_rf)}")

Random Forest Accuracy: 0.8191747572815534


In [34]:
# logistic regression:
from sklearn.linear_model import LogisticRegression

preprocessor = ColumnTransformer(
    transformers=[
        # to ensure the age does not have too high weightage
        ('num', MinMaxScaler(), ['age']),
        ('ord', OrdinalEncoder(categories=[ordinal_order]), ['education']),
        # need to drop a redundant column or else the model might break, suggested by AI
        ('nom', OneHotEncoder(drop='first', sparse_output=False), ['job', 'marital']),
        ('bin', OneHotEncoder(drop='if_binary'), ['default', 'housing', 'loan'])
    ])

features_processed_log = preprocessor.fit_transform(features)
X_train_log, X_test_log, y_train_log, y_test_log = train_test_split(features_processed_log, target, test_size=0.2)
scaler = StandardScaler()
X_train_scaled_log = scaler.fit_transform(X_train_log)
X_test_scaled_log = scaler.fit_transform(X_test_log)
# We can use the same scaled data from above
params2 = {
    'C' : [0.00001, 0.0001, 0.001, 0.01, 0.1, 1],
    'penalty' : ['l1'], 
    'max_iter' : [1000], 
    'solver': ['liblinear'],
}
grid_search2 = GridSearchCV(LogisticRegression(class_weight='balanced'), params2)
grid_search2.fit(X_train_scaled_log, y_train_log)
print(f'The best hyperparams are: {grid_search2.best_params_}')

The best hyperparams are: {'C': 1e-05, 'max_iter': 1000, 'penalty': 'l1', 'solver': 'liblinear'}


In [35]:
log_reg = LogisticRegression(C=0.00001, max_iter=1000, penalty='l1', solver='liblinear', class_weight='balanced')
log_reg.fit(X_train_log, y_train_log)
print(f'The accuracy is for logistic regression is: {log_reg.score(X_test_log, y_test_log)}')

The accuracy is for logistic regression is: 0.8992718446601942


d) Compare using train, validation and test set split versus using cross-validation. Which one performs better? (1p)

In [43]:
print('d KNN')
from sklearn.model_selection import StratifiedKFold
from sklearn.model_selection import cross_val_score

# stratifiedKFold is used because there was a user warning when doing the normal cross validation
# it was likely due to an uneven split of yes no in the target variable.
# You can track multiple metrics simultaneously
skf = StratifiedKFold(shuffle=True, random_state=1)
scores = cross_val_score(KNeighborsClassifier(), features_processed, target, cv=skf)
print(f'CV score: {scores.mean()}')
print('prev test split accuracy score: 0.8786407766990292')

d KNN
CV score: 0.8820096379572722
prev test split accuracy score: 0.8786407766990292


In [45]:
print('d randomforest')
skf = StratifiedKFold(shuffle=True, random_state=1)
scores = cross_val_score(RandomForestClassifier(), features_processed_tree, target, cv=skf)
print(f'CV score: {scores.mean()}')
print('prev test split accuracy score: 0.8191747572815534')

d randomforest
CV score: 0.8611308379242413
prev test split accuracy score: 0.8191747572815534


In [48]:
print('d logistic regression')
skf = StratifiedKFold(shuffle=True, random_state=1)
scores = cross_val_score(LogisticRegression(), features_processed_log, target, cv=skf)
print(f'CV score: {scores.mean()}')
print('prev test split accuracy score: 0.8992718446601942')

d logistic regression
CV score: 0.8905074378605388
prev test split accuracy score: 0.8992718446601942


For part D, cross validation performed better for random forest and KNN, but it performed worst for logistic regression

e) Report and evaluate the performance of the models using several of the metrics provided in the course, and explain which model is the best for the task and why. (1p)

Since the difference between cross validation and split training is rather small, we will stick with the split training models.

In [46]:
from sklearn.metrics import classification_report

print('KNN')
y_pred_k = knn.predict(X_test_scaled_k)
print(classification_report(y_test_k, y_pred_k))
print(f"The accuracy from the model is is 0.8786407766990292.")

KNN
              precision    recall  f1-score   support

          no       0.88      0.99      0.94       727
         yes       0.33      0.03      0.06        97

    accuracy                           0.88       824
   macro avg       0.61      0.51      0.50       824
weighted avg       0.82      0.88      0.83       824

The accuracy from the model is is 0.8786407766990292.


In [47]:
print('random forest')
y_pred_rf = rf.predict(X_test_rf)
print(classification_report(y_test_rf, y_pred_rf))
print(f"The accuracy from the model is is 0.8191747572815534")

random forest
              precision    recall  f1-score   support

          no       0.89      0.91      0.90       728
         yes       0.18      0.16      0.17        96

    accuracy                           0.82       824
   macro avg       0.54      0.53      0.53       824
weighted avg       0.81      0.82      0.81       824

The accuracy from the model is is 0.8191747572815534


In [49]:
print('logistic regression. I tried modifying some of the hyperparameters but I still could not get the logistic regression model to output yes')
y_pred_log = log_reg.predict(X_test_log)
print(classification_report(y_test_log, y_pred_log))
print(f"The accuracy from the model is is 0.8992718446601942")

logistic regression. I tried modifying some of the hyperparameters but I still could not get the logistic regression model to output yes
              precision    recall  f1-score   support

          no       0.90      1.00      0.95       741
         yes       0.00      0.00      0.00        83

    accuracy                           0.90       824
   macro avg       0.45      0.50      0.47       824
weighted avg       0.81      0.90      0.85       824

The accuracy from the model is is 0.8992718446601942


c:\Users\yyoda\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
c:\Users\yyoda\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
c:\Users\yyoda\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


For all three models they performed very well on predicting "no" as most of the dataset is no, but performed rather badly on predicting the yes as the precision and recall for the yes is rather low. For this campaign the most import factor would be the recall as it is what makes the bank money, the random forest model would be the best model out of the three for this.